# DeltaNet — a toy-scale build of the pure delta rule

A minimal implementation of **DeltaNet**, from Yang, Wang, Yu, Kim,
*"Parallelizing Linear Transformers with the Delta Rule over Sequence
Length"* (2024) — the direct architectural ancestor of KDA (see `../kda`),
isolating the **delta rule** on its own, with no decay gate at all.

Companion write-up: `README.md` in this folder.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The idea

This repo's `gla/` folder shows what happens when you add a **decay gate**
to plain linear attention: the state fades over time, controlled by the
input. This folder shows the *other* ingredient KDA combines with decay: the
**delta rule** — on its own, with the decay gate removed entirely.

Plain linear attention (see `../linear-attention`) only ever *adds* new
key-value pairs to its state, which means if the same key shows up twice
with two different values, the state ends up holding a blend of both,
forever. The delta rule fixes this differently than a decay gate does: at
every step, it explicitly **removes whatever was previously stored for this
specific key**, before writing the new value in.

```
kv_old = k_t^T S_{t-1}                          # what the state currently associates with this key
S_t = S_{t-1} - beta_t * k_t (x) kv_old + beta_t * k_t (x) v_t     # erase the old association, write the new one
o_t = q_t^T S_t
```

Note there's no `alpha_t * S_{t-1}` decay term anywhere — the state is only
ever modified at the exact key being written to, not uniformly faded
everywhere. This is why it's called the "delta rule": the update is
proportional to the *difference* (delta) between what the state currently
says about this key and what it should say — `beta_t` controls how much of
that difference to correct on this step (a learning-rate-like quantity, not
a decay).

**How this compares to KDA:** KDA (`../kda`) takes exactly this delta-rule
update and adds a channel-wise decay gate on top (the `alpha_t * S_{t-1}`
term this folder omits). DeltaNet is what's left with that decay removed —
a good place to see the delta rule in isolation, and to feel out what the
decay gate actually buys you when you compare training on the same toy task.

> **Simplification used here:** the paper's main contribution is a
> **chunkwise-parallel algorithm** for computing this update efficiently on
> a GPU (delta-rule updates are trickier to parallelize than plain decay,
> since each step's erase operation depends on the exact state produced by
> the previous step). This notebook uses the plain sequential recurrence —
> a Python loop over timesteps — same math, much easier to read.

In [ ]:
class ShortConv(nn.Module):
    def __init__(self, dim, kernel_size=4):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv1d(dim, dim, kernel_size, groups=dim, padding=0)
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.pad(x, (self.kernel_size - 1, 0))
        return self.conv(x).transpose(1, 2)

In [ ]:
class DeltaNet(nn.Module):
    def __init__(self, d_model=64, n_heads=2, d_head=32, conv_kernel=4):
        super().__init__()
        self.h, self.dh = n_heads, d_head
        inner = n_heads * d_head
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.q_conv = ShortConv(inner, conv_kernel)
        self.k_conv = ShortConv(inner, conv_kernel)
        self.beta_proj = nn.Linear(d_model, n_heads, bias=True)   # correction strength, not a decay
        self.gate_proj = nn.Linear(d_model, inner, bias=True)
        self.out_proj = nn.Linear(inner, d_model, bias=False)
        self.out_norm = nn.LayerNorm(d_head)

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.h, self.dh
        q = F.silu(self.q_conv(self.q_proj(x))).view(B, T, H, Dh)
        k = F.silu(self.k_conv(self.k_proj(x))).view(B, T, H, Dh)
        v = self.v_proj(x).view(B, T, H, Dh)
        q = F.normalize(q, p=2, dim=-1)
        k = F.normalize(k, p=2, dim=-1)
        beta = torch.sigmoid(self.beta_proj(x))                    # B,T,H

        S = x.new_zeros(B, H, Dh, Dh)
        outs = []
        for t in range(T):                                          # sequential delta-rule recurrence
            k_t, v_t, q_t, b_t = k[:, t], v[:, t], q[:, t], beta[:, t]
            kv_old = torch.einsum('bhd,bhde->bhe', k_t, S)             # what's currently stored for this key
            S = S - b_t.view(B, H, 1, 1) * k_t.unsqueeze(-1) * kv_old.unsqueeze(-2)   # erase it
            S = S + b_t.view(B, H, 1, 1) * k_t.unsqueeze(-1) * v_t.unsqueeze(-2)      # write the new value
            o_t = torch.einsum('bhd,bhde->bhe', q_t, S)
            outs.append(o_t)

        o = self.out_norm(torch.stack(outs, dim=1)).reshape(B, T, H * Dh)
        gate = torch.sigmoid(self.gate_proj(x))
        return self.out_proj(gate * o)

## 2. Assembling a tiny language model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([DeltaNet(d_model) for _ in range(n_layers)])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through DeltaNet once it's wired into a real model. So the rest of this
notebook:

1. wraps DeltaNet into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Add the decay gate back** and you've built KDA (`../kda`) — try training
  both on the same toy task and compare how quickly the loss drops; the
  decay gate gives the model an extra way to control what stays in the
  state that the delta rule alone doesn't provide.
- **Compare against plain Linear Attention** (`../linear-attention`), which
  has neither decay nor the delta rule — the "no gating at all" baseline
  this whole family of architectures builds on top of.
- **Try the chunkwise-parallel form** from the paper — computing the
  delta-rule recurrence in parallel is genuinely trickier than for a
  plain-decay layer like GLA, since each timestep's erase operation depends
  on the exact state left by the previous step; the paper's algorithm
  works around this with a clever reformulation worth reading closely.

Reference: Yang, Wang, Yu, Kim, *"Parallelizing Linear Transformers with the
Delta Rule over Sequence Length,"* 2024.